In [2]:
import sys
sys.argv = ['']

In [5]:
import torch
import argparse
from torch_geometric.datasets import WikipediaNetwork, TUDataset, Planetoid, Coauthor, CitationFull, QM9
from utils import load_graph_data, coarsening_classification, coarsening_regression, colater 
from torch.utils.data import DataLoader as T_DataLoader
from network import Classify_graph_gc, Classify_graph_gs, Regress_graph_gc, Regress_graph_gs
from run import graph_infer_Gs

def arg_correction(args):
    if args.super_graph:
        args.cluster_node = False
        args.extra_node = False
    elif args.cluster_node:
        args.extra_node = False
        args.super_graph = False
    elif args.extra_node:
        args.cluster_node = False
        args.super_graph = False
    return args

def process_dataset(args):
    # Node Classification
    if args.dataset == 'dblp':
        dataset = CitationFull(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'Physics':
        dataset = Coauthor(root='./dataset/Physics', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'cora':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'citeseer':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    elif args.dataset == 'pubmed':
        dataset = Planetoid(root='./dataset', name=args.dataset)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_cls'
    #Node Regression
    elif args.dataset == 'chameleon':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    elif args.dataset == 'squirrel':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    elif args.dataset == 'crocodile':
        dataset = WikipediaNetwork(root='./dataset', name=args.dataset, geom_gcn_preprocess=False)
        if args.normalize_features:
            dataset.x = torch.nn.functional.normalize(dataset.x, p=1)
        args.task = 'node_reg'
    #Graph Classification
    elif args.dataset == 'ENZYMES':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        for i in range(len(dataset)):
            if args.normalize_features:
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_cls'
        args.num_classes = 6
    elif args.dataset == 'PROTEINS':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        for i in range(len(dataset)):
            if args.normalize_features:
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_cls'
    elif args.dataset == 'AIDS':
        dataset = TUDataset(root='./dataset', name=args.dataset)
        for i in range(len(dataset)):
            if args.normalize_features:
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_cls'
        args.num_classes = 2
    #Graph Regression
    elif args.dataset == 'QM9':
        dataset = QM9(root='./dataset/QM9')
        for i in range(len(dataset)):
            if args.normalize_features:
                dataset[i].x = torch.nn.functional.normalize(dataset[i].x, p=1)
        args.task = 'graph_reg'
    return dataset, args

parser = argparse.ArgumentParser()
parser.add_argument('--dataset', type=str, default='chameleon')
parser.add_argument('--experiment', type=str, default='fixed') #'fixed', 'random', 'few'
parser.add_argument('--runs', type=int, default=20)
parser.add_argument('--exp_setup', type=str, default='Gc_train_2_Gs_infer') # 'Gc_train_2_Gs_infer', 'Gs_train_2_Gs_infer'
parser.add_argument('--hidden', type=int, default=512)
parser.add_argument('--epochs1', type=int, default=100)
parser.add_argument('--epochs2', type=int, default=300)
parser.add_argument('--num_layers1', type=int, default=2)
parser.add_argument('--num_layers2', type=int, default=2)
parser.add_argument('--batch_size', type=int, default=128)
parser.add_argument('--train_ratio', type=float, default=0.3)
parser.add_argument('--val_ratio', type=float, default=0.2)
parser.add_argument('--early_stopping', type=int, default=10)
parser.add_argument('--extra_node', type=bool, default=False)
parser.add_argument('--cluster_node', type=bool, default=False)
parser.add_argument('--super_graph', type=bool, default=False)
parser.add_argument('--lr', type=float, default=0.01)
parser.add_argument('--weight_decay', type=float, default=0.0005)
parser.add_argument('--normalize_features', type=bool, default=True)
parser.add_argument('--coarsening_ratio', type=float, default=0.5)
parser.add_argument('--coarsening_method', type=str, default='variation_neighborhoods') #'variation_neighborhoods', 'variation_edges', 'variation_cliques', 'heavy_edge', 'algebraic_JC', 'affinity_GS', 'kron'
parser.add_argument('--task', type = str, default = 'node_cls')         ### node_reg, graph_cls, graph_reg
parser.add_argument('--seed', type = int, default = None)               ### Seed for reproducibility
parser.add_argument('--property', type = int, default = 0)              ### Property for graph regression task
args = parser.parse_args()

args.dataset = "AIDS"                                                           ### Enter dataset name here      
args = arg_correction(args)
dataset, args = process_dataset(args)

index = 1                                                                       ### Index for single graph

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "model.pt"                                             #           ## Enter model name here
path  = "./save/graph_cls/AIDS_Gc_train_2_Gs_infer_0.1_affinity_GS_128_0.001/"  ### Add path here
new_dataset = []
if args.task == "graph_cls":
    args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_classification(args, dataset[index], 1-args.coarsening_ratio, args.coarsening_method)
    Gc = load_graph_data(dataset[index], CLIST, GcLIST, candidate)
    Gs = subgraph_list
    new_dataset.append((dataset[index], Gc, Gs))

    colater_fn = colater()
    test_loader = T_DataLoader(new_dataset, batch_size=1, collate_fn=colater_fn)

    model_gs = Classify_graph_gs(args).to(device)
    loss_fn = torch.nn.CrossEntropyLoss().to(device)
    model_gs.load_state_dict(torch.load(path + model_name))
    model_gs.eval()
    for batch in test_loader:
        set_gs = batch[1]
        y = batch[2].to(device).type(torch.long)
        batch_tensor = batch[3].to(device)
        out = model_gs(set_gs, batch_tensor)
        loss = loss_fn(out, y)
    print(f"Loss: {loss.item()}\nTrue|Predicted: {y.item()}|{out.argmax().item()}")
    print(f"Probs: {out}")

elif args.task == "graph_reg":

    args.num_features, candidate, C_list, Gc_list, subgraph_list, component_2_subgraphs, CLIST, GcLIST = coarsening_regression(args, dataset[index], 1-args.coarsening_ratio, args.coarsening_method)
    Gc = load_graph_data(dataset[index], CLIST, GcLIST, candidate)
    Gs = subgraph_list
    new_dataset.append((dataset[index], Gc, Gs))

    colater_fn = colater()
    test_loader = T_DataLoader(new_dataset, batch_size=1, collate_fn=colater_fn)

    model_gs = Regress_graph_gs(args).to(device)
    loss_fn = torch.nn.MSELoss().to(device)
    model_gs.load_state_dict(torch.load(path + model_name))
    model_gs.eval()
    for batch in test_loader:
        set_gs = batch[1]
        y = batch[2].to(device).type(torch.long)
        batch_tensor = batch[3].to(device)
        out = model_gs(set_gs, batch_tensor)
        loss = loss_fn(out, y[:, args.property].view(-1, 1))
    print(f"Loss: {loss.item()}\nTrue|Predicted: {y[:, args.property].item()}|{out.argmax().item()}")

Loss: 0.4579620063304901
True|Predicted: 1|1
Probs: tensor([[0.2284, 0.7716]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
